# Limpeza de `pagamentos.csv`: do log às tabelas

O log tem uma cobrança por linha. Ele vira três tabelas em `dados/processados/`, nas mesmas chaves das tabelas de assinatura:

| Tabela | Uma linha por | Liga com |
|---|---|---|
| `pagamentos_limpo.parquet` | cobrança (sem as cópias) | `assinatura_id` |
| `pagamentos_mes.parquet` | assinatura × mês, as mesmas linhas do painel | `assinaturas_mes.parquet` por `assinatura_id` + `mes` |
| `pagamentos_resumo.parquet` | assinatura | `assinaturas_limpa.parquet` por `assinatura_id` |

**Regras aplicadas** (todas no `decisoes.md`):
1. `valor` em dois formatos (`34.90` e `R$ 34,90`) convertido para número.
2. A mesma cobrança aprovada gravada duas vezes, com `pagamento_id` seguido: sai a cópia, fica o menor `pagamento_id`.
3. Data em horário de Brasília (com `Z` é UTC, das lojas; sem `Z` já está em Brasília, do site).
4. `tentativa` não é usada: a nova tentativa é a cobrança aprovada até 16 dias depois de uma recusa.
5. Chargeback e estorno devolvem o valor da última cobrança aprovada, não o preço de tabela da linha.
6. Receita líquida com as taxas do enunciado: 15% na loja, 3,5% no cartão, 0,99% no Pix; numa devolução,
   a taxa volta junto (o enunciado não diz; hipótese registrada).

Nada em `dados/brutos/` é alterado: o hash é conferido no início e no fim.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
INICIO_LOG = pd.Timestamp("2024-07-01")
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)
TZ_BR = "America/Sao_Paulo"
JANELA_NOVA_TENTATIVA = pd.Timedelta(days=16)
TAXA = {"app_store": 0.15, "google_play": 0.15, "cartao": 0.035, "pix_automatico": 0.0099}
CUPOM = 239.90

ARQUIVOS = ["pagamentos.csv", "usuarios.csv", "eventos_assinatura.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

registro = []

def registrar(tabela, passo, n, obs=""):
    registro.append({"tabela": tabela, "passo": passo, "linhas": n, "observacao": obs})
    print(f"[{tabela}] {passo}: {n:,} linhas" + (f" ({obs})" if obs else ""))

## 0. Leitura

O log como texto; as tabelas de assinatura já limpas (notebooks 05 e 06).

In [2]:
pag = pd.read_csv(BRUTOS / "pagamentos.csv", dtype=str, keep_default_na=False)
usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str, keep_default_na=False)
assin = pd.read_parquet(PROCESSADOS / "assinaturas_limpa.parquet")
painel = pd.read_parquet(PROCESSADOS / "assinaturas_mes.parquet", columns=["assinatura_id", "mes"])
log_ev = pd.read_parquet(PROCESSADOS / "eventos_assinatura_limpo.parquet", columns=["assinatura_id", "evento", "data_br"])

assert len(pag) == 165_548   # cabeçalho da seção 07 do dicionário
assert pag["pagamento_id"].is_unique and not pag.duplicated().any()
assert pag["user_id"].isin(assin["user_id"]).all()

email = usuarios.set_index("user_id")["email"].str.strip().str.lower()
assert not pag["user_id"].map(email).str.endswith("@vitaliza.com.br").any()

registrar("log", "0. Arquivo bruto", len(pag), "0 de contas de teste; todo user_id tem assinatura")
pd.crosstab(pag["status"], pag["metodo"], margins=True, margins_name="total")

[log] 0. Arquivo bruto: 165,548 linhas (0 de contas de teste; todo user_id tem assinatura)


metodo,app_store,cartao,google_play,pix_automatico,total
status,,,,,
aprovado,56329,32845,54314,18442,161930
chargeback,96,111,89,0,296
estornado,21,23,28,0,72
recusado,574,1697,675,304,3250
total,57020,34676,55106,18746,165548


## 1. Valor como número

Todas as cobranças do site até 30/06/2025 vêm como `R$ 34,90`; as demais, como `34.90`.

In [3]:
br_fmt = pag["valor"].str.startswith("R$ ")
texto = pag["valor"].str.removeprefix("R$ ")
texto = texto.where(~br_fmt, texto.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
pag["valor_num"] = pd.to_numeric(texto, errors="coerce")
assert pag["valor_num"].notna().all()
print(f"formato R$ 0,00: {br_fmt.sum():,} linhas; falhas de conversão: 0")
display(pag.groupby(["status", "valor_num"]).size().rename("cobranças").to_frame())

sinal_ok = np.where(pag["status"].isin(["aprovado", "recusado"]), pag["valor_num"] > 0, pag["valor_num"] < 0)
assert sinal_ok.all()
registrar("log", "1. Valor como número", len(pag), f"{br_fmt.sum():,} no formato R$ 0,00")

formato R$ 0,00: 19,750 linhas; falhas de conversão: 0


cobranças
status     valor_num           
aprovado    34.9         101278
            39.9          43641
            239.9          1686
            299.9         15325
chargeback -299.9            90
           -39.9            206
estornado  -299.9            72
recusado    34.9           1976
            39.9            991
            239.9            24
            299.9           259

[log] 1. Valor como número: 165,548 linhas (19,750 no formato R$ 0,00)


## 2. Data em horário de Brasília

In [4]:
def para_brasilia(s):
    utc = s.str.endswith("Z")
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    out[utc] = (pd.to_datetime(s[utc], format="%Y-%m-%dT%H:%M:%SZ", utc=True)
                .dt.tz_convert(TZ_BR).dt.tz_localize(None))
    out[~utc] = pd.to_datetime(s[~utc], format="%Y-%m-%d %H:%M:%S")
    return out

pag["data_br"] = para_brasilia(pag["data_hora"])
assert pag["data_br"].notna().all()
assert (pag["data_hora"].str.endswith("Z") == pag["metodo"].isin(["app_store", "google_play"])).all()
assert pag["data_br"].min() >= INICIO_LOG and pag["data_br"].max() < FIM_CORTE
muda_dia = (pag["data_br"].dt.normalize() != pd.to_datetime(pag["data_hora"].str[:10])).sum()
registrar("log", "2. Data em Brasília", len(pag), f"{muda_dia:,} mudam de dia")

[log] 2. Data em Brasília: 165,548 linhas (12,871 mudam de dia)


## 3. Cobranças gravadas duas vezes

Mesma conta, instante, valor, método, status e tentativa; só o `pagamento_id` muda, e é o seguinte.

In [5]:
pag["id_num"] = pag["pagamento_id"].str.removeprefix("p_").astype("int64")
pag = pag.sort_values("id_num").reset_index(drop=True)
chave = ["user_id", "data_hora", "valor", "metodo", "status", "tentativa"]
copia = pag.duplicated(chave, keep="first")

original = pag.loc[pag.duplicated(chave, keep="last") & ~copia, chave + ["id_num"]]
pares = pag.loc[copia, chave + ["id_num"]].merge(original, on=chave, suffixes=("_copia", "_original"))
print(f"cópias: {copia.sum():,}, todas de status {pag.loc[copia, 'status'].unique().tolist()}; "
      f"distância entre os ids: {(pares['id_num_copia'] - pares['id_num_original']).unique().tolist()}; "
      f"R$ {pag.loc[copia, 'valor_num'].sum():,.2f}")
assert copia.sum() == 1_941 and (pag.loc[copia, "status"] == "aprovado").all()
assert (pares["id_num_copia"] - pares["id_num_original"] == 1).all()

pag = pag.loc[~copia].reset_index(drop=True)
registrar("log", "3. Sem as cópias", len(pag), f"{copia.sum():,} removidas")

cópias: 1,941, todas de status ['aprovado']; distância entre os ids: [1]; R$ 118,295.90
[log] 3. Sem as cópias: 163,607 linhas (1,941 removidas)


## 4. Assinatura, nova tentativa, devolução e cupom

- `assinatura_id`: cada conta tem uma assinatura (notebook 05).
- `nova_tentativa`: aprovada, e a cobrança anterior da conta é uma recusa de até 16 dias antes.
  `dias_desde_recusa` mede esse atraso.
- `valor_devolvido`: em chargeback e estorno, o valor da última cobrança aprovada da conta antes da devolução.
- `cupom`: cobrança de R$ 239,90 (anual com cupom de influenciador, só no primeiro ano).

In [6]:
pag["assinatura_id"] = pag["user_id"].map(assin.set_index("user_id")["assinatura_id"])
pag = pag.sort_values(["user_id", "data_br", "id_num"]).reset_index(drop=True)

anterior_status = pag.groupby("user_id")["status"].shift()
anterior_data = pag.groupby("user_id")["data_br"].shift()
desde = pag["data_br"] - anterior_data
pag["nova_tentativa"] = ((pag["status"] == "aprovado") & (anterior_status == "recusado")
                         & (desde <= JANELA_NOVA_TENTATIVA))
pag["dias_desde_recusa"] = (desde.dt.total_seconds() / 86400).round(2).where(pag["nova_tentativa"])

aprov_apos_recusa = (pag["status"] == "aprovado") & (anterior_status == "recusado")
print(f"aprovadas logo depois de uma recusa: {aprov_apos_recusa.sum():,}; até 16 dias: {pag['nova_tentativa'].sum():,} "
      f"(observado: {pag['dias_desde_recusa'].min()} a {pag['dias_desde_recusa'].max()} dias)")
display(pd.crosstab(pag["nova_tentativa"], pag["tentativa"].rename("tentativa (original)")))
assert pag["nova_tentativa"].sum() == aprov_apos_recusa.sum() == 1_881

aprovadas logo depois de uma recusa: 1,881; até 16 dias: 1,881 (observado: 0.32 a 15.45 dias)


tentativa (original),1,2
nova_tentativa,,
False,156968,4758
True,1827,54


In [7]:
devol = pag["status"].isin(["chargeback", "estornado"])
aprovadas = pag.loc[pag["status"] == "aprovado", ["user_id", "data_br", "valor_num"]].rename(
    columns={"data_br": "data_cobranca", "valor_num": "valor_cobrado"}).sort_values("data_cobranca")
d = pag.loc[devol, ["user_id", "data_br"]].reset_index().sort_values("data_br")
d = pd.merge_asof(d, aprovadas, left_on="data_br", right_on="data_cobranca", by="user_id",
                  direction="backward").set_index("index")
assert d["valor_cobrado"].notna().all()

pag["valor_devolvido"] = d["valor_cobrado"].reindex(pag.index)
difere = devol & (pag["valor_devolvido"] != -pag["valor_num"])
print(f"devoluções: {devol.sum()}; com valor de tabela diferente do cobrado: {difere.sum()} "
      f"(R$ {(-pag.loc[difere, 'valor_num'] - pag.loc[difere, 'valor_devolvido']).sum():,.2f} a mais na linha negativa)")
display(pd.crosstab([pag.loc[devol, "status"], -pag.loc[devol, "valor_num"]], pag.loc[devol, "valor_devolvido"]))
assert difere.sum() == 179

devoluções: 368; com valor de tabela diferente do cobrado: 179 (R$ 3,150.00 a mais na linha negativa)


valor_devolvido       34.9   39.9   239.9  299.9
status     valor_num                            
chargeback 39.9         138     68      0      0
           299.9          0      0     35     55
estornado  299.9          0      0      6     66

In [8]:
pag["cupom"] = pag["valor_num"].abs().round(2) == CUPOM
plano = pag["assinatura_id"].map(assin.set_index("assinatura_id")["plano_atual"])
assert (plano[pag["cupom"]] == "anual").all()

aprov = pag.loc[pag["status"] == "aprovado"].copy()
aprov["ordem"] = aprov.groupby("user_id").cumcount() + 1
contas_cupom = set(aprov.loc[(aprov["ordem"] == 1) & aprov["cupom"], "user_id"])
cupom_aprov = aprov.loc[aprov["cupom"]]
print(f"cobranças com cupom: {pag['cupom'].sum():,} ({len(cupom_aprov):,} aprovadas, "
      f"{(pag['cupom'] & (pag['status'] == 'recusado')).sum()} recusadas); contas que entraram com cupom: {len(contas_cupom):,}")
display(cupom_aprov["ordem"].value_counts().sort_index().rename("aprovadas com cupom, por ordem da cobrança na conta").to_frame())
assert cupom_aprov["user_id"].isin(contas_cupom).all()   # só quem entrou com cupom paga com cupom

cobranças com cupom: 1,688 (1,664 aprovadas, 24 recusadas); contas que entraram com cupom: 1,434


,"aprovadas com cupom, por ordem da cobrança na conta"
ordem,
1,1434
2,230


O enunciado diz que o cupom vale só no primeiro ano e que a renovação é cobrada pelo preço cheio. No log, a maioria
das renovações de quem entrou com cupom sai de novo a R$ 239,90, em todos os canais e trimestres. O valor cobrado
fica como está (é o que a pessoa pagou) e a coluna `cupom_na_renovacao` marca esses casos; a regra fica em aberto
com o dono do dado.

In [9]:
renov_cupom = aprov.loc[aprov["user_id"].isin(contas_cupom) & (aprov["ordem"] >= 2)]
display(pd.crosstab(renov_cupom["ordem"].rename("ordem da cobrança"), renov_cupom["valor_num"].rename("valor")))
segunda = renov_cupom.loc[renov_cupom["ordem"] == 2]
display(pd.crosstab(segunda["data_br"].dt.to_period("Q").rename("trimestre"), segunda["valor_num"].rename("valor")))
display(pd.crosstab(segunda["metodo"], segunda["valor_num"].rename("valor")))

pag["cupom_na_renovacao"] = pag["pagamento_id"].isin(cupom_aprov.loc[cupom_aprov["ordem"] >= 2, "pagamento_id"])
print(f"renovações cobradas com cupom: {pag['cupom_na_renovacao'].sum()} de {len(segunda)} primeiras renovações "
      f"(R$ {pag['cupom_na_renovacao'].sum() * (299.90 - CUPOM):,.2f} abaixo do preço cheio)")
assert pag["cupom_na_renovacao"].sum() == 230 and (cupom_aprov["ordem"] <= 2).all()

valor,239.9,299.9
ordem da cobrança,,
2,230,34
3,0,24


valor,239.9,299.9
trimestre,,
2025Q3,35,12
2025Q4,46,8
2026Q1,64,9
2026Q2,35,5
2026Q3,50,0


valor,239.9,299.9
metodo,,
app_store,83,16
cartao,48,5
google_play,78,8
pix_automatico,21,5


renovações cobradas com cupom: 230 de 264 primeiras renovações (R$ 13,800.00 abaixo do preço cheio)


## 5. Receita bruta e líquida

Por cobrança: aprovada soma o valor; recusada não soma; devolução desconta o valor devolvido.
A líquida tira a taxa do meio de pagamento (na devolução, a taxa volta junto).

In [10]:
pag["taxa"] = pag["metodo"].map(TAXA)
pag["receita_bruta"] = np.select([pag["status"] == "aprovado", devol],
                                 [pag["valor_num"], -pag["valor_devolvido"]], 0.0)
pag["receita_liquida"] = pag["receita_bruta"] * (1 - pag["taxa"])

exemplo = pd.DataFrame({"metodo": ["app_store", "cartao", "pix_automatico"], "valor": 39.90})
exemplo["líquido"] = (exemplo["valor"] * (1 - exemplo["metodo"].map(TAXA))).round(2)
exemplo["enunciado"] = [33.92, 38.50, 39.50]
display(exemplo)
assert (exemplo["líquido"] == exemplo["enunciado"]).all()
assert abs(299.90 * (1 - TAXA["app_store"]) - 254.92) < 0.006   # 254,915: o enunciado arredonda para cima

print(f"receita bruta: R$ {pag['receita_bruta'].sum():,.2f} | líquida: R$ {pag['receita_liquida'].sum():,.2f}")

,metodo,valor,líquido,enunciado
0,app_store,39.9,33.92,33.92
1,cartao,39.9,38.50,38.50
2,pix_automatico,39.9,39.50,39.50


receita bruta: R$ 10,104,367.90 | líquida: R$ 8,974,354.72


## 6. Conferência com o log de eventos

Toda cobrança aprovada é um início ou uma renovação, e vice-versa; toda devolução tem o evento correspondente.

In [11]:
pag["mes"] = pag["data_br"].dt.to_period("M").dt.to_timestamp()
ev = log_ev.dropna(subset=["assinatura_id"])
contagem_ev = ev["evento"].value_counts()

conf = pd.DataFrame([
    ("aprovadas = inícios + renovações", (pag["status"] == "aprovado").sum(),
     contagem_ev["assinatura_iniciada"] + contagem_ev["renovacao"]),
    ("chargeback = eventos chargeback", (pag["status"] == "chargeback").sum(), contagem_ev["chargeback"]),
    ("estornado = eventos reembolso", (pag["status"] == "estornado").sum(), contagem_ev["reembolso"]),
    ("recusadas com carência no mesmo dia (das carências)",
     pag.loc[pag["status"] == "recusado"].assign(dia=lambda x: x["data_br"].dt.normalize())
        .merge(ev.loc[ev["evento"] == "periodo_carencia"].assign(dia=lambda x: x["data_br"].dt.normalize()),
               on=["assinatura_id", "dia"])["assinatura_id"].nunique(),
     contagem_ev["periodo_carencia"]),
], columns=["conferência", "pagamentos", "eventos"]).set_index("conferência")
conf["bate"] = conf["pagamentos"] == conf["eventos"]
display(conf)
assert conf["bate"].all()

# aprovadas por assinatura e mês = inícios + renovações por assinatura e mês
a_mes = pag.loc[pag["status"] == "aprovado"].groupby(["assinatura_id", "mes"]).size()
e_mes = (ev.loc[ev["evento"].isin(["assinatura_iniciada", "renovacao"])]
         .assign(mes=lambda x: x["data_br"].dt.to_period("M").dt.to_timestamp())
         .groupby(["assinatura_id", "mes"]).size())
dif = a_mes.sub(e_mes, fill_value=0)
print(f"assinatura-mês em que aprovadas ≠ inícios + renovações: {(dif != 0).sum()}")
assert (dif == 0).all()

COLS_LOG = ["pagamento_id", "assinatura_id", "user_id", "data_br", "mes", "status", "metodo", "valor_num",
            "valor_devolvido", "cupom", "cupom_na_renovacao", "nova_tentativa", "dias_desde_recusa", "taxa", "receita_bruta",
            "receita_liquida", "tentativa", "data_hora", "valor"]
log = pag[COLS_LOG].rename(columns={"valor_num": "valor", "valor": "valor_original",
                                    "tentativa": "tentativa_original", "data_hora": "data_hora_original"})
registrar("log", "4-6. pagamentos_limpo.parquet", len(log))

,pagamentos,eventos,bate
conferência,,,
aprovadas = inícios + renovações,159989,159989,True
chargeback = eventos chargeback,296,296,True
estornado = eventos reembolso,72,72,True
recusadas com carência no mesmo dia (das carências),1346,1346,True


assinatura-mês em que aprovadas ≠ inícios + renovações: 0
[log] 4-6. pagamentos_limpo.parquet: 163,607 linhas


## 7. Tabela: pagamentos por mês (`pagamentos_mes.parquet`)

As mesmas linhas do painel (assinatura × mês); mês sem cobrança fica com zero.

In [12]:
g = pag.groupby(["assinatura_id", "mes"])
mes = pd.DataFrame({
    "n_aprovadas": g["status"].apply(lambda s: (s == "aprovado").sum()),
    "n_recusadas": g["status"].apply(lambda s: (s == "recusado").sum()),
    "n_novas_tentativas": g["nova_tentativa"].sum(),
    "n_devolucoes": g["status"].apply(lambda s: s.isin(["chargeback", "estornado"]).sum()),
    "receita_bruta": g["receita_bruta"].sum(),
    "receita_liquida": g["receita_liquida"].sum(),
    "valor_devolvido": g["valor_devolvido"].sum(),
    "max_dias_desde_recusa": g["dias_desde_recusa"].max(),
}).reset_index()

fora = mes.merge(painel, on=["assinatura_id", "mes"], how="left", indicator=True)["_merge"].eq("left_only").sum()
print(f"assinatura-mês com cobrança fora do painel: {fora}")
assert fora == 0

pm = painel.merge(mes, on=["assinatura_id", "mes"], how="left", validate="one_to_one")
cont = ["n_aprovadas", "n_recusadas", "n_novas_tentativas", "n_devolucoes"]
pm[cont] = pm[cont].fillna(0).astype("int64")
pm[["receita_bruta", "receita_liquida", "valor_devolvido"]] = pm[["receita_bruta", "receita_liquida", "valor_devolvido"]].fillna(0.0)
pm["teve_recusa"] = pm["n_recusadas"] > 0

assert len(pm) == len(painel)
assert pm[cont].sum().tolist() == [(pag["status"] == "aprovado").sum(), (pag["status"] == "recusado").sum(),
                                   pag["nova_tentativa"].sum(), devol.sum()]
assert np.isclose(pm["receita_bruta"].sum(), pag["receita_bruta"].sum())
registrar("mes", "7. Uma linha por linha do painel", len(pm),
          f"{(pm['n_aprovadas'] + pm['n_recusadas'] + pm['n_devolucoes'] > 0).sum():,} com alguma cobrança")

receita_mes = pm.groupby("mes")[["receita_bruta", "receita_liquida"]].sum().round(2)
display(receita_mes.T)

assinatura-mês com cobrança fora do painel: 0


[mes] 7. Uma linha por linha do painel: 327,771 linhas (159,169 com alguma cobrança)


mes,2024-07-01,2024-08-01,2024-09-01,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01,2025-08-01,2025-09-01,2025-10-01,2025-11-01,2025-12-01,2026-01-01,2026-02-01,2026-03-01,2026-04-01,2026-05-01,2026-06-01,2026-07-01,2026-08-01,2026-09-01
receita_bruta,221753.70,264672.20,283317.50,294436.60,295978.10,289357.10,364512.80,326815.50,359065.60,325486.40,329506.40,319095.80,362434.20,386273.80,410152.20,415381.10,409936.00,378574.9,507925.10,441384.40,492352.60,437721.0,423842.20,393262.40,422463.60,458793.20,489873.50
receita_liquida,196604.21,235034.08,252520.44,262379.44,263653.99,257511.29,323766.88,289701.15,319333.44,289249.42,293098.44,283186.92,321989.53,342857.92,364284.65,368938.74,364533.39,336564.9,450008.78,391544.97,437121.06,387983.8,376697.71,349579.71,374773.53,407390.89,434045.45


## 8. Tabela: resumo por assinatura (`pagamentos_resumo.parquet`)

Uma linha por assinatura. Para as anteriores ao log, os valores contam só de 01/07/2024 em diante.

In [13]:
ga = pag.groupby("assinatura_id")
res = pd.DataFrame({
    "n_aprovadas": ga["status"].apply(lambda s: (s == "aprovado").sum()),
    "n_recusadas": ga["status"].apply(lambda s: (s == "recusado").sum()),
    "n_novas_tentativas": ga["nova_tentativa"].sum(),
    "n_devolucoes": ga["status"].apply(lambda s: s.isin(["chargeback", "estornado"]).sum()),
    "receita_bruta": ga["receita_bruta"].sum(),
    "receita_liquida": ga["receita_liquida"].sum(),
    "valor_devolvido": ga["valor_devolvido"].sum(),
    "primeira_recusa": pag.loc[pag["status"] == "recusado"].groupby("assinatura_id")["data_br"].min(),
    "teve_cupom": ga["cupom"].any(),
    "cupom_na_renovacao": ga["cupom_na_renovacao"].any(),
    "valor_medio_cobranca": pag.loc[pag["status"] == "aprovado"].groupby("assinatura_id")["valor_num"].mean(),
})
res = (assin[["assinatura_id", "user_id", "inicio_antes_do_log"]].set_index("assinatura_id").join(res))
# sem cobrança no log: só assinaturas anteriores a ele que terminaram antes da próxima cobrança
sem_cobranca = res["n_aprovadas"].isna()
print(f"assinaturas sem nenhuma cobrança desde 01/07/2024: {sem_cobranca.sum()}, "
      f"todas anteriores ao log: {res.loc[sem_cobranca, 'inicio_antes_do_log'].all()}")
assert res.loc[sem_cobranca, "inicio_antes_do_log"].all()
res[cont] = res[cont].fillna(0).astype("int64")
res[["receita_bruta", "receita_liquida", "valor_devolvido"]] = res[["receita_bruta", "receita_liquida", "valor_devolvido"]].fillna(0.0)
res["teve_cupom"] = res["teve_cupom"].astype("boolean").fillna(False).astype(bool)
res["cupom_na_renovacao"] = res["cupom_na_renovacao"].astype("boolean").fillna(False).astype(bool)
res["teve_recusa"] = res["n_recusadas"] > 0
res["teve_devolucao"] = res["n_devolucoes"] > 0
res = res.reset_index()

assert len(res) == len(assin) and res["assinatura_id"].is_unique
assert np.isclose(res["receita_bruta"].sum(), pag["receita_bruta"].sum())
assert np.isclose(res["receita_liquida"].sum(), pm["receita_liquida"].sum())
registrar("resumo", "8. Uma linha por assinatura", len(res))
display(res.describe().T[["count", "mean", "min", "max"]].round(2))

assinaturas sem nenhuma cobrança desde 01/07/2024: 801, todas anteriores ao log: True
[resumo] 8. Uma linha por assinatura: 32,583 linhas


,count,mean,min,max
n_aprovadas,32583.0,4.910199,0.0,28.0
n_recusadas,32583.0,0.099745,0.0,4.0
n_novas_tentativas,32583.0,0.057729,0.0,4.0
n_devolucoes,32583.0,0.011294,0.0,1.0
receita_bruta,32583.0,310.11165,0.0,1017.2
receita_liquida,32583.0,275.430584,0.0,1007.12972
valor_devolvido,32583.0,1.646662,0.0,299.9
primeira_recusa,3013,2025-10-17 15:15:48.764686592,2024-07-02 09:11:44,2026-09-30 22:18:22
valor_medio_cobranca,31736.0,135.971128,34.9,299.9


## 9. Saídas e integridade dos brutos

In [14]:
saidas = {
    "pagamentos_limpo.parquet": log,
    "pagamentos_mes.parquet": pm,
    "pagamentos_resumo.parquet": res,
}
for nome, df in saidas.items():
    df = df.reset_index(drop=True)
    for c in df.select_dtypes("object").columns:
        df[c] = df[c].astype(object).where(df[c].notna(), None)
    df.to_parquet(PROCESSADOS / nome, index=False)
    pd.testing.assert_frame_equal(pd.read_parquet(PROCESSADOS / nome), df)
    print(f"dados/processados/{nome}: {len(df):,} linhas x {df.shape[1]} colunas")

pd.DataFrame(registro).to_csv(SAIDAS / "limpeza_pagamentos_registro.csv", index=False)
display(pd.DataFrame(registro))

hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/processados/pagamentos_limpo.parquet: 163,607 linhas x 19 colunas


dados/processados/pagamentos_mes.parquet: 327,771 linhas x 11 colunas
dados/processados/pagamentos_resumo.parquet: 32,583 linhas x 16 colunas


,tabela,passo,linhas,observacao
0,log,0. Arquivo bruto,165548,0 de contas de teste; todo user_id tem assinatura
1,log,1. Valor como número,165548,"19,750 no formato R$ 0,00"
2,log,2. Data em Brasília,165548,"12,871 mudam de dia"
3,log,3. Sem as cópias,163607,"1,941 removidas"
4,log,4-6. pagamentos_limpo.parquet,163607,
5,mes,7. Uma linha por linha do painel,327771,"159,169 com alguma cobrança"
6,resumo,8. Uma linha por assinatura,32583,


dados/brutos/ inalterado: True
